In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine, text

In [ ]:
# --- CẤU HÌNH ---
SERVER_NAME = 'LAPTOP-E0LRO698'
DB_STAGE = 'OnlineRetailDWStage'
# Sử dụng Trusted Connection (Windows Authentication)
conn_str = f'mssql+pyodbc://{SERVER_NAME}/{DB_STAGE}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
engine = create_engine(conn_str)

def extract_to_staging():
    run_id = int(datetime.now().strftime("%d%H%M%S"))
    print(f"Bắt đầu ETL Run ID: {run_id}")

    try:
        # 1. Đọc dữ liệu từ nguồn file
        # file nằm cùng thư mục với script
        sales_raw = pd.read_csv('online_retail_2.csv')
        country_map = pd.read_excel('country_mapping.xlsx')

        # 2. Tiền xử lý nhanh để khớp kiểu dữ liệu SQL
        # Đổi tên cột cho khớp với bảng Sales_Stage đã tạo ở trên
        sales_df = sales_raw.rename(columns={
            'Invoice': 'InvoiceNo',
            'Description': 'ProductName',
            'Price': 'UnitPrice',
            'Customer ID': 'CustomerID'
        })
        
        # Chuyển đổi InvoiceDate sang chuẩn datetime của SQL
        sales_df['InvoiceDate'] = pd.to_datetime(sales_df['InvoiceDate'])
        
        # 3. Nạp vào Staging (Sử dụng 'replace' để làm mới dữ liệu mỗi lần chạy)
        with engine.begin() as conn:
            # Xóa dữ liệu cũ trong Stage trước khi nạp mới
            conn.execute(text("TRUNCATE TABLE Sales_Stage"))
            conn.execute(text("TRUNCATE TABLE CountryMap_Stage"))
            
            # Nạp dữ liệu mới
            sales_df.to_sql('Sales_Stage', con=conn, if_exists='append', index=False)
            country_map.to_sql('CountryMap_Stage', con=conn, if_exists='append', index=False)
            
            # Ghi Log thành công
            log_query = text("""
                INSERT INTO ETL_LOG_Stage (RunID, TableName, StartTime, EndTime, RowsLoaded, Status)
                VALUES (:rid, 'All_Stage', :start, :end, :rows, 'Success')
            """)
            conn.execute(log_query, {
                "rid": run_id,
                "start": datetime.now(),
                "end": datetime.now(),
                "rows": len(sales_df) + len(country_map)
            })

        print(f"--- Hoàn thành nạp Staging ---")
        print(f"Sales: {len(sales_df)} rows")
        print(f"Country Map: {len(country_map)} rows")

    except Exception as e:
        print(f"Lỗi trong quá trình Extract: {e}")

if __name__ == "__main__":
    extract_to_staging()